<!-- NB_DOC_INTRO_V1 -->
# Data Engineering — Extraction de documents avec Docling

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Docling** (IBM Research, 2024) = lib moderne pour **extraire le contenu structure** de PDF / HTML / DOCX / images : texte, tables, figures, hierarchie de sections. Equivalent open-source de LlamaParse. Ideal pour pipelines RAG (transforme un PDF en chunks structures avec context).

Note : Docling necessite une installation lourde (~500 MB avec modeles vision-language). Ce notebook execute du **pseudo-code** + alternatives plus legeres (pypdf, pdfplumber, unstructured).

## Plan

1. Concepts (Docling vs autres)
2. Pseudo-code Docling end-to-end
3. Alternatives legeres (pypdf, pdfplumber, pymupdf)
4. Chunking pour RAG
5. Hybrid : texte + table + image
6. Pieges + References


In [ ]:
import warnings
warnings.filterwarnings("ignore")
print("Docling : lib heavy — code reference uniquement.")
print("Pour exec reelle : uv pip install docling")

## 1. Concepts — pourquoi Docling

Extraire le texte d'un PDF est **trivial avec `pypdf`**. Extraire la **structure** (titres, tables, ordre de lecture multi-colonnes, captions d'images) est dur.

Docling utilise des **modeles vision-language** pour :
- Detecter la **structure hierarchique** (titres, sections, paragraphes)
- Extraire les **tables** comme objets (rowspan/colspan preserves)
- Identifier les **figures** avec captions
- Reconstruire le **reading order** (multi-colonnes, sidebars)
- Gerer les **formules math**

Resultat : **JSON structure** prêt pour RAG (chunks avec contexte hierarchique) ou conversion en Markdown / HTML.

| Tool | Texte | Tables | Layout | OCR | Vitesse | Cas |
|---|---|---|---|---|---|---|
| **pypdf** | ✅ basique | ❌ | ❌ | ❌ | ⚡⚡⚡ | Extraction texte simple |
| **pdfplumber** | ✅ | ✅ basique | partial | ❌ | ⚡⚡ | Tables simples |
| **pymupdf** (fitz) | ✅ | ✅ | partial | ❌ | ⚡⚡⚡ | Texte + images, rapide |
| **unstructured** | ✅ | ✅ | ✅ | ✅ | ⚡ | All-in-one mais lourd |
| **Docling** (IBM) | ✅ | ✅ ++ | ✅ ++ | ✅ | 🐢 (gros modele) | RAG production-grade |
| **LlamaParse** (commercial) | ✅ ++ | ✅ ++ | ✅ ++ | ✅ | API | Si budget |


## 2. Docling — pseudo-code end-to-end

```python
# pip install docling
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("paper.pdf")

doc = result.document
print(doc.export_to_markdown())   # markdown structure
# ou
print(doc.export_to_json())       # JSON detaille
```

### Structure de la sortie

```python
for item in doc.iterate_items():
    print(f"Type : {item.label}")           # 'title', 'section_header', 'paragraph', 'table', 'figure'
    print(f"Text : {item.text[:100]}")
    if hasattr(item, "captions"):
        print(f"Caption : {item.captions}")
```

### Chunking pour RAG

```python
from docling.chunking import HybridChunker

chunker = HybridChunker()
chunks = chunker.chunk(doc)

for chunk in chunks:
    print(f"--- Chunk (size={len(chunk.text)} chars) ---")
    print(f"Headings : {chunk.meta.headings}")    # contexte hierarchique conserve
    print(chunk.text[:200])
```

Le **HybridChunker** preserve la **hierarchie** : un chunk d'un paragraphe garde le titre de section comme metadata → bien meilleur retrieval.


## 3. Alternatives legeres — pypdf, pdfplumber, pymupdf

In [ ]:
# pip install pypdf
example_pypdf = '''
from pypdf import PdfReader

reader = PdfReader("doc.pdf")
print(f"Nb pages : {len(reader.pages)}")
print(f"Metadata : {reader.metadata}")

for page in reader.pages:
    text = page.extract_text()
    print(text[:500])
'''
print("=== pypdf (texte simple, rapide) ===")
print(example_pypdf)

In [ ]:
# pip install pdfplumber (tables)
example_pdfplumber = '''
import pdfplumber

with pdfplumber.open("doc.pdf") as pdf:
    for page in pdf.pages:
        # Texte
        text = page.extract_text()

        # Tables (bonne extraction)
        tables = page.extract_tables()
        for table in tables:
            df = pd.DataFrame(table[1:], columns=table[0])
            print(df)

        # Images
        for img in page.images:
            print(f"Image at {img['x0']}, {img['top']}")
'''
print("=== pdfplumber (texte + tables) ===")
print(example_pdfplumber)

In [ ]:
# pip install pymupdf (le plus complet et rapide)
example_pymupdf = '''
import fitz   # PyMuPDF

doc = fitz.open("doc.pdf")
print(f"Pages : {len(doc)}")

for page in doc:
    text = page.get_text()           # texte
    blocks = page.get_text("blocks")  # texte par block (avec positions)
    images = page.get_images()        # images
    tables = page.find_tables()       # tables (PyMuPDF >= 1.23)

    # Render en image (pour OCR ulterieure)
    pix = page.get_pixmap(dpi=200)
    pix.save(f"page_{page.number}.png")
'''
print("=== pymupdf / fitz (le plus complet) ===")
print(example_pymupdf)

## 4. Chunking strategies pour RAG

| Strategie | Description | Quand |
|---|---|---|
| **Fixed-size** | N caracteres avec overlap | Baseline |
| **Sentence-based** | Aux fins de phrases | Preserve structure |
| **Recursive** (LangChain) | `\n\n` → `\n` → `. ` → ` ` | Default LangChain |
| **Semantic** | Embeddings + detection rupture sens | Experimental, lent |
| **Hierarchical** | Section → Paragraph → Sentence, hierarchy preserved | Pour RAG production |
| **MarkdownHeader** | Decoupe sur `#`, `##`, ... | Pour docs markdown structures |

Docling fait du **HybridChunker** = hierarchique + tailleur dynamique.

## 5. Pipeline RAG complet

```
PDF/DOCX/HTML
    ↓
[Docling / pdfplumber / pymupdf]
    ↓
Documents structures (texte + tables + figures + hierarchie)
    ↓
[Chunker (HybridChunker / Recursive)]
    ↓
Chunks (avec metadata : section, page, page_num)
    ↓
[Embedder (sentence-transformers)]
    ↓
[Vector DB (Chroma / Qdrant)]
    ↓
Recherche : query → top-k chunks → LLM
```

## 6. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `pypdf` pour PDF avec tables complexes | pdfplumber / pymupdf / Docling |
| Pas garder la metadata page/section | Tres utile pour citations sources |
| Chunks sans contexte hierarchique | HybridChunker preserve les headings |
| Skip OCR sur PDF scannes | Tesseract / unstructured |
| Pas dedupliquer les headers/footers | Polluent les chunks |
| Toujours mettre PDF en raw text | Tables doivent rester structurees (markdown ou JSON) |
| Charger toute la lib pour 1 page | Au moins fitz est leger |


## References

### Documentation
- Docling : https://ds4sd.github.io/docling/
- pdfplumber : https://github.com/jsvine/pdfplumber
- PyMuPDF : https://pymupdf.readthedocs.io/
- unstructured : https://docs.unstructured.io/
- LlamaParse (commercial) : https://www.llamaindex.ai/llamaparse

### Voir aussi
- [NLP_LangChain_RAG.ipynb](NLP_LangChain_RAG.ipynb)
- [retrieval_BDD_Vectorielle.ipynb](retrieval_BDD_Vectorielle.ipynb)
- [BDD_Vectorielles.ipynb](BDD_Vectorielles.ipynb)
